# 01 — Exploratory Data Analysis: Consumo Eléctrico (AEP/PJM)

Análisis exploratorio del dataset de consumo eléctrico horario de la región AEP (American Electric Power) de PJM Interconnection.

**Objetivo:** Entender la estructura temporal del consumo antes de modelar.

**Dataset:** `data/raw/AEP_hourly.csv` — descargar desde [Kaggle](https://www.kaggle.com/datasets/robikscube/hourly-energy-consumption)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.figsize"] = (14, 4)

## 1. Carga y Vista General

In [ ]:
df = pd.read_csv("../data/raw/AEP_hourly.csv")
df.columns = ["datetime", "aep_mw"]
df["datetime"] = pd.to_datetime(df["datetime"])
df = df.sort_values("datetime").reset_index(drop=True)

print(f"Filas: {len(df):,}")
print(f"Rango: {df['datetime'].min()} → {df['datetime'].max()}")
print(f"Años cubiertos: {df['datetime'].dt.year.nunique()}")
df.head()

In [ ]:
df["aep_mw"].describe().round(1)

## 2. Serie de Tiempo Completa

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(df["datetime"], df["aep_mw"], linewidth=0.4, color="steelblue", alpha=0.8)
ax.set_title("Consumo Eléctrico Horario — AEP/PJM", fontsize=14)
ax.set_xlabel("Fecha")
ax.set_ylabel("Consumo (MW)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()

## 3. Calidad de Datos

In [ ]:
# Valores faltantes
print("=== Valores nulos ===")
print(df.isnull().sum())

# Duplicados en timestamp
dup = df.duplicated(subset="datetime").sum()
print(f"\nTimestamps duplicados: {dup}")

# Huecos en la serie horaria
expected = pd.date_range(df["datetime"].min(), df["datetime"].max(), freq="h")
missing_hours = expected.difference(df["datetime"])
print(f"Horas faltantes en el rango: {len(missing_hours)}")

## 4. Distribución del Consumo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histograma
axes[0].hist(df["aep_mw"], bins=60, color="steelblue", edgecolor="white", linewidth=0.3)
axes[0].set_title("Distribución del Consumo")
axes[0].set_xlabel("MW")
axes[0].set_ylabel("Frecuencia")

# Boxplot por año
df["year"] = df["datetime"].dt.year
df.boxplot(column="aep_mw", by="year", ax=axes[1], 
           boxprops=dict(color="steelblue"),
           medianprops=dict(color="tomato", linewidth=2))
axes[1].set_title("Distribución por Año")
axes[1].set_xlabel("Año")
axes[1].set_ylabel("MW")
plt.suptitle("")
plt.tight_layout()
plt.show()

## 5. Patrones Estacionales

In [ ]:
df["hour"] = df["datetime"].dt.hour
df["dayofweek"] = df["datetime"].dt.dayofweek
df["month"] = df["datetime"].dt.month

day_labels = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
month_labels = ["Ene", "Feb", "Mar", "Abr", "May", "Jun",
                "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# Por hora del día
hourly_avg = df.groupby("hour")["aep_mw"].mean()
axes[0].bar(hourly_avg.index, hourly_avg.values, color="steelblue", edgecolor="white")
axes[0].set_title("Consumo Promedio por Hora del Día")
axes[0].set_xlabel("Hora")
axes[0].set_ylabel("MW")
axes[0].set_xticks(range(24))

# Por día de la semana
dow_avg = df.groupby("dayofweek")["aep_mw"].mean()
colors = ["steelblue"] * 5 + ["tomato"] * 2
axes[1].bar(range(7), dow_avg.values, color=colors, edgecolor="white")
axes[1].set_title("Consumo Promedio por Día de la Semana")
axes[1].set_xlabel("Día")
axes[1].set_ylabel("MW")
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(day_labels)

plt.tight_layout()
plt.show()

In [ ]:
# Por mes
fig, ax = plt.subplots(figsize=(14, 4))
monthly_avg = df.groupby("month")["aep_mw"].mean()
ax.bar(range(1, 13), monthly_avg.values, color="steelblue", edgecolor="white")
ax.set_title("Consumo Promedio por Mes")
ax.set_xlabel("Mes")
ax.set_ylabel("MW")
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap hora × día de la semana
pivot = df.pivot_table(values="aep_mw", index="hour", columns="dayofweek", aggfunc="mean")
pivot.columns = day_labels

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(
    pivot, 
    cmap="YlOrRd", 
    ax=ax,
    fmt=".0f",
    linewidths=0.3,
    cbar_kws={"label": "MW"}
)
ax.set_title("Heatmap: Consumo Promedio por Hora y Día", fontsize=13)
ax.set_xlabel("Día de la Semana")
ax.set_ylabel("Hora del Día")
plt.tight_layout()
plt.show()

## 6. Tendencia Año a Año

In [ ]:
# Media móvil de 30 días para ver tendencia
df_daily = df.set_index("datetime")["aep_mw"].resample("D").mean()
trend = df_daily.rolling(window=30, center=True).mean()

fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(df_daily.index, df_daily.values, alpha=0.3, color="steelblue", linewidth=0.8, label="Diario")
ax.plot(trend.index, trend.values, color="tomato", linewidth=2, label="Tendencia (30d)")
ax.set_title("Consumo Diario con Tendencia de 30 días")
ax.set_xlabel("Fecha")
ax.set_ylabel("MW")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Descomposición Estacional

In [ ]:
# Usar datos diarios para la descomposición (menos ruido que horario)
decomp = seasonal_decompose(df_daily.dropna(), model="additive", period=365, extrapolate_trend="freq")

fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)

decomp.observed.plot(ax=axes[0], color="steelblue", linewidth=0.6)
axes[0].set_ylabel("Observado")

decomp.trend.plot(ax=axes[1], color="tomato", linewidth=1.2)
axes[1].set_ylabel("Tendencia")

decomp.seasonal.plot(ax=axes[2], color="seagreen", linewidth=0.6)
axes[2].set_ylabel("Estacionalidad")

decomp.resid.plot(ax=axes[3], color="gray", linewidth=0.5)
axes[3].set_ylabel("Residuos")

axes[0].set_title("Descomposición Estacional (Aditiva, periodo=365d)", fontsize=13)
plt.tight_layout()
plt.show()

## 8. Autocorrelación (ACF / PACF)

La ACF y PACF confirman qué lags son más relevantes como features para el modelo.

In [ ]:
# Usar sample horario (últimos 60 días para que sea legible)
sample = df.set_index("datetime")["aep_mw"].last("60D").dropna()

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

plot_acf(sample, lags=48, ax=axes[0], color="steelblue")
axes[0].set_title("ACF — Autocorrelación (hasta lag 48h)")
axes[0].set_xlabel("Lag (horas)")

plot_pacf(sample, lags=48, ax=axes[1], color="tomato", method="ywm")
axes[1].set_title("PACF — Autocorrelación Parcial (hasta lag 48h)")
axes[1].set_xlabel("Lag (horas)")

plt.tight_layout()
plt.show()

## Conclusiones del EDA

| Hallazgo | Implicación para el modelo |
|---|---|
| Patrón horario fuerte (valle nocturno, pico tarde) | Incluir `hour` como feature |
| Caída clara en fines de semana | Incluir `is_weekend` / `dayofweek` |
| Estacionalidad anual (picos verano/invierno) | Lags semanales y rolling mensual |
| Alta autocorrelación en lags 1h, 24h y 168h | Justifica los lags del feature engineering |
| Tendencia decreciente suave en el largo plazo | Posible mejora: añadir variable de tendencia |